# 14a What the fMRI map draws on, and P-7 over images as well as people

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 10a and 10b.

Development analysis, not a prespecified test. Part 1: own ridge against the map on four subspaces of 200 or 1,000 dimensions (the donors' prediction
directions, feature PCA, a random projection, all features), each started from borrowing or from nothing. Part 2: a
two-way bootstrap (people and image blocks) of P-7's three claims.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '14a-subspaceboot', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"K": 200, "folds": 5, "n_galleries": 4, "gallery_size": 200, "gallery_seed": 346, "delta_r_share": 0.05, "delta_top1": 2.0, "n_claims": 3, "alpha": 0.05, "roi_classes": ["prf-visualrois", "floc-faces", "floc-places", "floc-bodies", "floc-words"], "ks": [250, 500], "draws": [13, 14, 15], "subspaces": ["donor", "featpca", "random", "full"], "random_seed": 416, "n_blocks": 20, "block_seed": 415, "n_boot": 2000, "boot_seed": 417, "wall_hours": 11.0}')
SUBJECTS = ['subj01', 'subj02', 'subj03', 'subj04', 'subj05', 'subj06', 'subj07', 'subj08']

import numpy as np
import pandas as pd
from cross_subject_encoding.eval import lowrank_map as lm
from cross_subject_encoding.eval import nsd_confirm as nc
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C

if not isinstance(FIXED["K"], int):
    raise RuntimeError("K is not fixed: fill it in from 09q before the confirmation runs")
IN = Path("/kaggle/input")
conv = next(p.parent for p in IN.rglob("shared_train_nsd_ids.npy"))
shared = np.load(conv / "shared_train_nsd_ids.npy")
cols, common = {}, {}
for h in ("lh", "rh"):
    masks = [np.load(next((conv / f"{s}_roi").rglob(f"{h}.all-vertices_fsaverage_space.npy"))) for s in SUBJECTS]
    common[h], cl = nc.common_vertices(masks)
    cols[h] = dict(zip(SUBJECTS, cl))
V = sum(len(common[h]) for h in common)
ids, Y = {}, {}
for s in SUBJECTS:
    ids[s] = np.load(conv / f"{s}_train_nsd_ids.npy")
    Y[s] = np.concatenate([np.load(conv / f"{s}_{h}_fmri.npy", mmap_mode="r")[:, cols[h][s]] for h in ("lh", "rh")], 1)
noise = np.concatenate([np.concatenate([np.load(conv / f"{s}_{h}_noise_ceiling.npy").astype(np.float64)[cols[h][s]] for h in ("lh", "rh")]) for s in SUBJECTS])
DELTA_R = FIXED["delta_r_share"] * float(np.median(np.sqrt(np.clip(noise, 0, None) / 100)))
FEATS = {}
for name in ("alexnet", "dinov2"):
    f = np.load(next(IN.rglob(f"10b_{name}_kpca1000.npz")))
    FEATS[name] = (f["z"].astype(np.float64), {int(i): j for j, i in enumerate(f["nsd_ids"])})
feats = lambda name, nsd_ids: FEATS[name][0][[FEATS[name][1][int(i)] for i in nsd_ids]]
test_ids = np.sort(shared)
rows_of = {s: {int(i): j for j, i in enumerate(ids[s])} for s in SUBJECTS}
test_rows = {s: np.array([rows_of[s][int(i)] for i in test_ids]) for s in SUBJECTS}
pool_rows = {s: np.setdiff1d(np.arange(len(ids[s])), test_rows[s]) for s in SUBJECTS}
GALS = nc.galleries(len(test_ids), FIXED["n_galleries"], FIXED["gallery_size"], FIXED["gallery_seed"])
rois = {}
for cls in FIXED["roi_classes"]:
    for s in SUBJECTS:
        lab = np.concatenate([np.load(next((conv / f"{s}_roi").rglob(f"{h}.{cls}_challenge_space.npy")))[cols[h][s]] for h in ("lh", "rh")])
        rois[(s, cls)] = lab
print(f"{V} common vertices; delta_r {DELTA_R:.4f}; {len(test_ids)} test images; pools", {s: len(pool_rows[s]) for s in SUBJECTS},
      "|", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
def ols_coef(x, y):
    # the paper's OLS (intercept, normal equations), per output, in float64
    xa = np.concatenate([np.ones((len(x), 1)), x], 1)
    return np.linalg.solve(xa.T @ xa, xa.T @ np.asarray(y, np.float64)).astype(np.float32)
ols_pred = lambda beta, x: (np.concatenate([np.ones((len(x), 1)), x], 1) @ beta.astype(np.float64))
BETA = {}
for name in ("alexnet",):
    for s in SUBJECTS:
        BETA[(name, s)] = ols_coef(feats(name, ids[s][pool_rows[s]]), Y[s][pool_rows[s]])
    print(name, "encoders fitted |", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
def oof(fit_predict, k, seed):
    split = np.array_split(np.random.default_rng(seed).permutation(k), FIXED["folds"])
    out = None
    for held in split:
        pred = fit_predict(np.setdiff1d(np.arange(k), held), held)
        if out is None:
            out = np.empty((k, pred.shape[1]))
        out[held] = pred
    return out

def hits(pred, meas, centred=False):
    # per gallery, whether each image's response correlates best with its own prediction (nc.identify_top1, per image)
    out = []
    for g in GALS:
        p, m = np.asarray(pred[g], np.float64), np.asarray(meas[g], np.float64)
        if centred:
            p, m = p - p.mean(0), m - m.mean(0)
        p = (p - p.mean(1, keepdims=True)) / (p.std(1, keepdims=True) + 1e-30)
        m = (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-30)
        out.append((m @ p.T).argmax(1) == np.arange(len(g)))
    return np.stack(out)  # (galleries, 200)

N_TEST = len(test_ids)
BLOCK = np.random.default_rng(FIXED["block_seed"]).permutation(N_TEST) % FIXED["n_blocks"]
W = np.random.default_rng(FIXED["boot_seed"]).multinomial(FIXED["n_blocks"], np.full(FIXED["n_blocks"], 1 / FIXED["n_blocks"]),
                                                          size=FIXED["n_boot"]).astype(np.float64)  # (boot, blocks)
W_IMG = W[:, BLOCK]  # each image's weight in each bootstrap

def boot_enc_r(pred, meas):
    # mean over vertices of the correlation across images, for every bootstrap weighting, from per-block sums
    n_b = np.bincount(BLOCK, minlength=FIXED["n_blocks"]).astype(np.float64)
    S = {}
    for name, arr in (("p", pred), ("m", meas), ("pp", pred * pred), ("mm", meas * meas), ("pm", pred * meas)):
        S[name] = np.stack([arr[BLOCK == b].sum(0) for b in range(FIXED["n_blocks"])])  # (blocks, V)
    out = np.empty(FIXED["n_boot"])
    for c in range(0, FIXED["n_boot"], 100):
        w = W[c:c + 100]
        n = (w @ n_b)[:, None]
        sp, sm = w @ S["p"], w @ S["m"]
        cov = w @ S["pm"] - sp * sm / n
        vp, vm = w @ S["pp"] - sp ** 2 / n, w @ S["mm"] - sm ** 2 / n
        out[c:c + 100] = (cov / np.sqrt(np.clip(vp * vm, 1e-30, None))).mean(1)
    return out

def boot_top1(h):
    # h (galleries, 200) hits; image weights from the bootstrap blocks; mean over galleries of the weighted hit rate
    vals = []
    for gi, g in enumerate(GALS):
        w = W_IMG[:, g]
        vals.append((w * h[gi]).sum(1) / w.sum(1) * 100)
    return np.mean(vals, 0)

rows, boot = [], {}
stopped = None
for p, s in enumerate(SUBJECTS, start=1):
    t = time.monotonic()
    donors = [d for d in SUBJECTS if d != s]
    meas = np.asarray(Y[s][test_rows[s]], np.float64)
    y_pool = np.asarray(Y[s][pool_rows[s]], np.float32)
    x_pool, x_test = feats("alexnet", ids[s][pool_rows[s]]), feats("alexnet", test_ids)
    acc_pool, acc_test = np.zeros((len(x_pool), V), np.float32), np.zeros((len(x_test), V), np.float32)
    for d in donors:  # 10c's running sum
        acc_pool += ols_pred(BETA[("alexnet", d)], x_pool).astype(np.float32)
        acc_test += ols_pred(BETA[("alexnet", d)], x_test).astype(np.float32)
    avg_pool, avg_test = acc_pool / len(donors), (acc_test / len(donors)).astype(np.float64)
    del acc_pool, acc_test
    mean_b, B = lm.prediction_basis(avg_pool, FIXED["K"])
    # the other subspaces, from features only (the person's unlabelled pool images)
    fm = x_pool.mean(0)
    _, _, vt = np.linalg.svd(x_pool - fm, full_matrices=False)
    Q = np.linalg.qr(np.random.default_rng([FIXED["random_seed"], p]).standard_normal((x_pool.shape[1], FIXED["K"])))[0]
    proj = {"donor": lambda a, x: (a - mean_b) @ B, "featpca": lambda a, x: (x - fm) @ vt[:FIXED["K"]].T,
            "random": lambda a, x: (x - fm) @ Q, "full": lambda a, x: x}
    def add(method, k, d, pred, centred=False):
        h = hits(pred, meas, centred)
        rows.append({"subject": s, "method": method, "k": k, "draw": d, "centred": centred,
                     "enc_r": nc.encoding_r(pred, meas), "top1": float(h.mean() * 100)})
        return h
    h = add("avg", 0, "-", avg_test, centred=True)
    boot[(s, "avg_c_top1")] = boot_top1(h)
    boot[(s, "avg_enc")] = boot_enc_r(avg_test, meas)
    add("avg", 0, "-", avg_test)
    for d in FIXED["draws"]:
        order = np.random.default_rng([d, p]).permutation(len(pool_rows[s]))
        for k in FIXED["ks"]:
            idx = order[:k]
            y_k, x_k, a_k = y_pool[idx].astype(np.float64), x_pool[idx], avg_pool[idx].astype(np.float64)
            preds = {}
            for sub in FIXED["subspaces"]:
                f_k, f_t = proj[sub](a_k, x_k), proj[sub](avg_test, x_test)
                for start in ("borrow", "none"):
                    base_k, base_t = (a_k, avg_test) if start == "borrow" else (0.0, 0.0)
                    model = RidgePerOutput(lm.ALPHAS, FIXED["folds"], d).fit(f_k, y_k - base_k)
                    preds[f"{sub}|{start}"] = base_t + model.predict(f_t)
            # out-of-fold predictions for the stack, exactly as 10c: the map refitted with its own cross-validation,
            # own ridge refitted with its chosen penalties
            preds["oof|donor|borrow"] = oof(lambda tr, he: lm.apply_lowrank_map(lm.fit_lowrank_map(a_k[tr], y_k[tr], mean_b, B, seed=d),
                                                                                a_k[he], mean_b, B), k, d)
            own = RidgePerOutput(lm.ALPHAS, FIXED["folds"], d).fit(x_k, y_k)
            preds["oof|full|none"] = oof(lambda tr, he: own.refit(x_k[tr], y_k[tr]).predict(x_k[he]), k, d)
            w = nc.stack_fit([preds["oof|donor|borrow"], preds["oof|full|none"]], y_k)
            stack = nc.stack_apply(w, [preds["oof|donor|borrow"], preds["oof|full|none"]],
                                   [preds["donor|borrow"], preds["full|none"]], y_k.mean(0))
            for name, pr in preds.items():
                if not name.startswith("oof|"):
                    add(name, k, d, pr)
            h_stack = add("stack", k, d, stack)
            h_own = hits(preds["full|none"], meas)
            boot[(s, "map_enc", k, d)] = boot_enc_r(preds["donor|borrow"], meas)
            boot[(s, "stack_top1", k, d)] = boot_top1(h_stack)
            boot[(s, "own_top1", k, d)] = boot_top1(h_own)
        if (time.monotonic() - T0) / 3600 > FIXED["wall_hours"]:
            stopped = f"wall clock after {s} draw {d}"
            break
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / "14a_scores.csv", index=False)
    print(s, {r["method"]: round(r["enc_r"], 4) for r in rows if r["subject"] == s and r["k"] == 500 and r["draw"] == 13},
          f"| {time.monotonic() - t:.0f} s |", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)
    if stopped:
        print("STOPPED:", stopped)
        break

In [ ]:
frame = pd.DataFrame(rows)
done = [s for s in SUBJECTS if (frame.subject == s).any()]
def per_subject(method, metric, centred=False, ks=None):
    f = frame[(frame.method == method) & (frame.centred == centred)]
    if ks is not None:
        f = f[f.k.astype(str).isin([str(k) for k in ks])]
    return f.groupby("subject")[metric].mean().reindex(done).to_numpy()
def desc(x):
    return {"mean": float(np.mean(x)), "positive": int((x > 0).sum()), "n": int(len(x))}
res = {"post_hoc": "development: subspace control and two-way bootstrap",
       "levels": {}, "contrasts": {}, "bootstrap": {}}
methods = sorted(set(frame.method) - {"avg"})
for m in methods + ["avg"]:
    res["levels"][m] = {"enc_r": float(np.mean(per_subject(m, "enc_r"))), "top1": float(np.mean(per_subject(m, "top1")))}
res["levels"]["avg_centred"] = {"top1": float(np.mean(per_subject("avg", "top1", centred=True)))}
pairs = [("donor|borrow", "donor|none"), ("donor|borrow", "featpca|borrow"), ("donor|borrow", "random|borrow"),
         ("donor|borrow", "full|borrow"), ("donor|borrow", "full|none"), ("full|borrow", "full|none"),
         ("donor|none", "full|none"), ("featpca|borrow", "random|borrow"), ("donor|borrow", "avg")]
for a, b in pairs:
    for metric in ("enc_r", "top1"):
        res["contrasts"][f"{a} - {b}|{metric}"] = desc(per_subject(a, metric) - per_subject(b, metric))
# check: 10c's numbers (C1 map - avg r +0.0511, C3 stack - own +3.72 top-1)
res["check_10c"] = {"C1": float(np.mean(per_subject("donor|borrow", "enc_r") - per_subject("avg", "enc_r"))),
                    "C3": float(np.mean(per_subject("stack", "top1") - per_subject("full|none", "top1")))}
# two-way bootstrap of P-7's claims: per person and bootstrap, the claim's difference averaged over k and draws as P-7
DELTA = {"C1": DELTA_R, "C2": FIXED["delta_top1"], "C3": FIXED["delta_top1"]}
kd = [(k, d) for k in FIXED["ks"] for d in FIXED["draws"]]
per = {"C1": np.stack([np.mean([boot[(s, "map_enc", k, d)] for k, d in kd], 0) - boot[(s, "avg_enc")] for s in done]),
       "C2": np.stack([np.mean([boot[(s, "stack_top1", k, d)] for k, d in kd], 0) - boot[(s, "avg_c_top1")] for s in done]),
       "C3": np.stack([np.mean([boot[(s, "stack_top1", k, d)] - boot[(s, "own_top1", k, d)] for k, d in kd], 0) for s in done])}
rng = np.random.default_rng(FIXED["boot_seed"] + 1)
q = FIXED["alpha"] / FIXED["n_claims"]
for c, x in per.items():  # x (people, boot)
    people_idx = rng.integers(0, len(done), size=(FIXED["n_boot"], len(done)))
    two_way = x[people_idx, np.arange(FIXED["n_boot"])[:, None]].mean(1)
    images_only = x.mean(0)
    res["bootstrap"][c] = {"margin": DELTA[c], "mean": float(x.mean()),
                           "two_way_lower": float(np.quantile(two_way, q)), "two_way_upper": float(np.quantile(two_way, 1 - q)),
                           "images_only_lower": float(np.quantile(images_only, q)),
                           "verdict_two_way": "supported" if np.quantile(two_way, q) > DELTA[c] else
                                              ("rejected" if np.quantile(two_way, 1 - q) < DELTA[c] else "inconclusive")}
res["stopped"] = stopped
(C.METRICS_DIR / "14a_summary.json").write_text(json.dumps(res, indent=2))
print("14A_SUMMARY_JSON " + json.dumps(res))